In [ ]:
from langgraph.graph import MessagesState, StateGraph, END
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.types import Command, interrupt
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
from langchain_tavily import TavilySearch
from langchain.messages import AIMessage, HumanMessage,ToolMessage
from langgraph.checkpoint.memory import MemorySaver

In [2]:
memory=MemorySaver()

In [3]:
tavily_search=TavilySearch()
tools=[tavily_search]

In [4]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
llm_with_tools=llm.bind_tools(tools=tools)
llm_bouncer = ChatGoogleGenerativeAI(model="gemini-3-flash-preview")

In [5]:
prompt_template=ChatPromptTemplate([
    ("system","You are an AI Agent. You should not Hallucinate, If you do not know an answer use tools to find answers."),
    MessagesPlaceholder(variable_name="messages"),
    ("system","Do not use tool unnecessarely. Try to anwer user query using the previous messages."),
])

prompt_template_bouncer=ChatPromptTemplate([
    ("system","You are an AI Agent. You ought to strictly output only two outputs 'DONE' and 'REVISE'."),
    MessagesPlaceholder(variable_name="messages"),
    ("system","Go through the human input and understand if the user is satisfied with the output or not. If the used does "
     "not wish to make any changes output DONE else output REVISE"),
])

In [ ]:
responder_chain=prompt_template | llm_with_tools
bouncer_chain=prompt_template_bouncer | llm_bouncer

def responder_node(state:MessagesState)->Command:
    current_state=state
    result=responder_chain.invoke(current_state)
    if isinstance(result,AIMessage) and hasattr(result,"tool_calls") and len(result.tool_calls)>0:
        return Command(
            goto="tool_executer",
            update={"messages":[result]}
        )
    else:
        return Command(
            goto="feedback_receiver",
            update={"messages":[result]}
        )

def feedback_node(state:MessagesState)->Command:
    lastAIMessage=state["messages"][-1]
    feedback=interrupt(
        value={
            "Generated Post":lastAIMessage.content[0]["text"],
            # "Query":"Are you satisfied with the generated post?"
        }
    )
    if "done" in feedback.lower():
        return Command(
            goto=END
        )
    else:
        return Command(
            goto="responder",
            update={"messages":[HumanMessage(content=feedback)]}
        )

def tool_node(state:MessagesState)->Command:
    lastAIMessage=state["messages"][-1]
    tool_results=[]
    for tool_calls in lastAIMessage.tool_calls:
        tool_name=tool_calls["name"]
        tool_args=tool_calls["args"]
        tool_id=tool_calls["id"]
        tool_to_execute=next((tool for tool in tools if tool.name==tool_name),None)

        if tool_to_execute:
            tool_result=tool_to_execute.invoke(tool_args)
            tool_results.append(ToolMessage(content=tool_result,tool_call_id=tool_id ))
    
    return Command(
        goto="responder",
        update={"messages":tool_results}
    )

In [7]:
config={
    "configurable":{
        "thread_id":1
    }
}

graph=StateGraph(MessagesState)
graph.add_node("responder",responder_node)
graph.add_node("tool_executer",tool_node)
graph.add_node("feedback_receiver",feedback_node)
graph.set_entry_point("responder")

app=graph.compile(checkpointer=memory)

In [ ]:
user_input=input("USER: ")
for chunk in app.stream({"messages":HumanMessage(content=user_input)}, config=config):
    print(chunk)

{'responder': {'messages': [AIMessage(content=[{'type': 'text', 'text': "Here's a 100-word LinkedIn post on AI as the new future:\n\n---\n\n**The Future is Now: Embracing the AI Revolution!**\n\nArtificial Intelligence is rapidly reshaping our world, moving beyond science fiction to become an indispensable part of our daily lives and industries. From automating complex tasks to providing unprecedented insights, AI is not just a tool; it's the engine driving the next era of innovation.\n\nWe're witnessing a paradigm shift across healthcare, finance, education, and beyond. Companies and professionals who embrace AI will unlock new efficiencies, foster creativity, and discover solutions to previously intractable problems.\n\nLet's lean into this transformative technology. What excites you most about the future with AI? Share your thoughts below!\n\n#AI #ArtificialIntelligence #Innovation #FutureTech #DigitalTransformation", 'extras': {'signature': 'Ct0BAQw51seQBaGirXduP+PwFXRCzdzx06U2nK+l

In [15]:
while True:
    snapshot=app.get_state(config=config)
    if snapshot.tasks and snapshot.tasks[0].interrupts:
        post = snapshot.tasks[0].interrupts[0].value['Generated Post']
        print(f"******GENERATED POST********\n{post}\nAre you satisfied with the generated post?")
        user_feedback=input("USER: ")
        feedback=bouncer_chain.invoke({"messages":[HumanMessage(content=user_feedback)]})
        print(feedback)
        if "done" in feedback.content[0]["text"].lower():
            app.invoke(Command(resume="done"),config=config)
        else:
            app.invoke(Command(resume=user_feedback),config=config)
    else:
        break

******GENERATED POST********
Here's a 100-word LinkedIn post on AI as the new future:

---

**The Future is Now: Embracing the AI Revolution!**

Artificial Intelligence is rapidly reshaping our world, moving beyond science fiction to become an indispensable part of our daily lives and industries. From automating complex tasks to providing unprecedented insights, AI is not just a tool; it's the engine driving the next era of innovation.

We're witnessing a paradigm shift across healthcare, finance, education, and beyond. Companies and professionals who embrace AI will unlock new efficiencies, foster creativity, and discover solutions to previously intractable problems.

Let's lean into this transformative technology. What excites you most about the future with AI? Share your thoughts below!

#AI #ArtificialIntelligence #Innovation #FutureTech #DigitalTransformation
Are you satisfied with the generated post?
content=[{'type': 'text', 'text': 'REVISE', 'extras': {'signature': 'EoYDCoMDAQw

In [16]:
snapshot=app.get_state(config=config)
# if snapshot.tasks and snapshot.tasks[0].interrupts:
print(snapshot.values["messages"][-1].content[0]["text"])


Here's a funnier version of the LinkedIn post on AI:

---

**Hold onto Your Circuits: AI Just Crashed the Party and It's Bringing the Future!**

Remember when AI was just for supervillains and sassy robots in movies? Well, surprise! It's here, it's real, and it's probably already doing your boring spreadsheets while you're still deciding on coffee.

AI isn't just "the future"; it's the future that arrived early, ate all the snacks, and is now optimizing everything from your Netflix recommendations to global logistics. It's like having a super-smart, slightly over-caffeinated intern who never sleeps and *actually* loves data.

So, let's not fight it. Let's embrace our new overlords... I mean, *assistants*! What's the funniest (or most terrifyingly accurate) AI prediction you've heard?

#AI #ArtificialIntelligence #FutureIsNow #RobotUprising #JustKidding #SortOf
